# 🎓 Capstone Project: The "Deep-Dive" Academic Research Assistant
**Created by:** Mohd Shabir  
**Course:** Google x Kaggle AI Agents Intensive Course 2025

---

## 🚀 The Problem
As a university student, keeping up with the latest methodologies, industry news, and credible sources is overwhelming. A simple search often returns surface-level articles, while deep-diving into complex topics can be dense and time-consuming. I needed a way to get a **comprehensive, technical overview** of a subject without spending hours manually collating sources and structuring notes.

---

## 💡 The Solution
I built a **Sequential Multi-Agent System** that automates this research pipeline. Instead of a single bot trying to do everything, I designed a team of specialized agents that mimic a real-world research workflow:

* **🧠 The Strategist (Planner Agent):** First, this agent analyzes my broad topic and breaks it down into specific, answerable research questions. It stops me from getting generic answers by forcing a structured approach.
* **🕵️ The Investigator (Researcher Agent):** This agent takes the plan and executes it. Utilizing an integrated Google Search tool, it actively browses the web to gather recent developments, technical details, and credible sources for each specific question.
* **✍️ The Synthesizer (Editor Agent):** Finally, this agent takes the raw, messy notes and compiles them into a clean, structured report complete with citations that I can actually use for my studies.

---

## 🛠️ Tech Stack
* **Framework:** Google Agent Development Kit (ADK)
* **Model:** Gemini 2.5 Flash Lite - selected for its speed and cost-efficiency in multi-turn workflows.
* **Architecture:** Sequential Workflow (Plan -> Research -> Edit).
* **Tools:** Integrated `Google Search` via the ADK.

In [ ]:
# --- STEP 1: INSTALLATION ---
# Installing the necessary libraries.
# 'google-adk' is the core framework for building the agents.

!pip install -Uq google-adk

print("✅ Environment ready.")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 26.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.9/319.9 kB 11.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-translate 3.12.1 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.19.5, but you have protobuf 5.29.5 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
pydrive2 1.21.3 requires cryptography<44, but you have cryptography 46.0.3 which is in

In [ ]:
# ── Step 2: Configure the environment and define tools ────────────────────────

import os
import arxiv
from kaggle_secrets import UserSecretsClient

# Importing the specific ADK components I need for this architecture
from google.adk.models.google_llm import Gemini
from google.adk.tools import google_search

# 1. Secure Authentication
# I'm loading my API key from Kaggle Secrets to keep it secure.
try:
    os.environ["GOOGLE_API_KEY"] = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    print("✅ Connected to Gemini API.")
except:
    print("❌ Error: API Key missing. Check Add-ons -> Secrets.")

# 2. Model Selection
# I'm using Flash Lite because it's fast enough for this logic-heavy task 
# and efficiently handles the context window needed for passing notes between agents.
model = Gemini(model="gemini-2.5-flash-lite")

# 3. Custom Tool Logic
# The standard Google Search is great for news, but for my data science coursework,
# I need actual papers. I built this wrapper around the ArXiv API to give
# my agent direct access to scientific literature.
def search_arxiv(query: str) -> str:
    """
    Queries the ArXiv database for academic papers.
    I've limited it to the top 2 results to keep the context window manageable
    while still providing high-quality references.
    """
    client = arxiv.Client()
    
    search = arxiv.Search(
        query=query,
        max_results=2,
        sort_by=arxiv.SortCriterion.Relevance
    )
    
    results = []
    for r in client.results(search):
        # Cleaning the summary text to ensure the LLM parses it easily
        clean_summary = r.summary.replace("\n", " ")[:300]
        results.append(f"📄 Paper: {r.title}\n📝 Summary: {clean_summary}...\n🔗 Link: {r.pdf_url}\n---")
    
    return "\n".join(results)

print("✅ Custom 'search_arxiv' tool loaded and ready.")

✅ Connected to Gemini API.
✅ Custom 'search_arxiv' tool loaded and ready.


In [ ]:
# ── Step 3: Define the three specialist agents ────────────────────────────────

from google.adk.agents import Agent

# --- AGENT 1: THE PLANNER ---
#    Receives a broad topic and outputs exactly 3 numbered research questions.
#    It has no tools — it only reasons. The strict "numbered list only" output
#    format makes it easy for the Researcher to parse and act on.

planner = Agent(
    name="planner",
    model=model,
    instruction="""
    You are a Strategic Research Lead.
    Take the user's broad topic and break it down into 3 distinct, targeted research questions.
    Output ONLY a numbered list.
    """
)

# --- AGENT 2: THE RESEARCHER ---
#    Receives the Planner's 3 questions and answers each one using Google Search.
#    google_search is a built-in ADK tool that grounds responses in live web
#    results, reducing hallucination. The instruction emphasises recent
#    developments and technical depth to ensure the output is substantive.

researcher = Agent(
    name="researcher",
    model=model,
    # Only use google_search to ensure stability
    tools=[google_search], 
    instruction="""
    You are an Academic Researcher.
    For EACH question, use `Google Search` to find high-quality information.
    Focus on recent developments and technical details.
    Compile detailed notes.
    """
)

# --- AGENT 3: THE EDITOR ---
#    Receives the Researcher's raw notes and reformats them into a structured
#    blog post with a title, strategy summary, key findings, and references.
#    It has no tools — its job is pure synthesis and writing.

editor = Agent(
    name="editor",
    model=model,
    instruction="""
    You are a Data Science Journal Editor.
    Synthesize the researcher's raw notes into a polished report.
    Structure:
    1. Professional Title
    2. 🎯 Research Strategy
    3. 💡 Key Findings
    4. 📚 References
    """
)

print("✅ Agent Team defined (Stable Mode).")

✅ Agent Team defined (Stable Mode).


In [ ]:
# ── Step 4: Assemble the SequentialAgent pipeline ─────────────────────────────

from google.adk.agents import SequentialAgent
from google.adk.runners import InMemoryRunner

# --- ARCHITECTURE: SEQUENTIAL PIPELINE ---
# I chose a Sequential architecture because this task has a strict dependency chain:
# We can't research without a plan, and we can't edit without research.
# The output of each agent automatically becomes the context for the next.
academic_team = SequentialAgent(
    name="academic_team_pipeline",
    sub_agents=[planner, researcher, editor]
)

# --- Step 5: RUNNING THE PIPELINE ---
# This topic is directly relevant to my current Data Science coursework.
my_topic = "How are AI Agents transforming Data Science workflows in 2025?"

print(f"🚀 Launching Research Pipeline on: '{my_topic}'")
print("⏳ The agents are now collaborating... (Planner -> Researcher -> Editor)")

# Initializing the runner to manage the session and execution loop
runner = InMemoryRunner(agent=academic_team)

# Executing the workflow. 'run_debug' is useful here to see the internal
# thought process (traces) of the agents as they work.
final_report = await runner.run_debug(my_topic)

print("\n" + "="*40)
print("🎓 FINAL GENERATED REPORT")
print("="*40)
print(final_report)

🚀 Launching Research Pipeline on: 'How are AI Agents transforming Data Science workflows in 2025?'
⏳ The agents are now collaborating... (Planner -> Researcher -> Editor)

 ### Created new session: debug_session_id

User > How are AI Agents transforming Data Science workflows in 2025?
planner > 1. What specific AI agent capabilities (e.g., automated data cleaning, feature engineering, model selection, hyperparameter tuning) are demonstrating the most significant impact on reducing time-to-insight in data science projects?
2. How are data scientists collaborating with AI agents in 2025, and what new roles or skillsets are emerging as a result of this human-agent partnership?
3. What are the primary challenges and ethical considerations (e.g., bias amplification, interpretability, security, job displacement) associated with the widespread adoption of AI agents in data science workflows, and how are organizations addressing them?
researcher > AI agents are significantly transforming data 